# AndinaLog 03B | Integración temporal IoT + eventos de flota

El objetivo es enriquecer la Evidencia 3 sin multiplicar filas ni usar información futura.

## Regla de unión

Para cada lectura se consideran solamente eventos del mismo `camion_id` con `timestamp_evento < timestamp_lectura`. Los conteos se calculan en ventanas de 60 minutos, 180 minutos y 24 horas.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ENTORNO = "auto"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"

def encontrar_raiz():
    if ENTORNO == "drive" or (ENTORNO == "auto" and "google.colab" in sys.modules):
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(RUTA_PROYECTO_DRIVE)
    else:
        raiz = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "proyecto-integrador/03_EDA/salidas_v3/andinalog_03b_evidencia_3_lecturas.csv").is_file()), None)
    if raiz is None:
        raise FileNotFoundError("No se encontró la raíz del proyecto")
    return raiz

RAIZ = encontrar_raiz()
RUTA_IOT = RAIZ / "proyecto-integrador/03_EDA/salidas_v3/andinalog_03b_evidencia_3_lecturas.csv"
RUTA_EVENTOS = RAIZ / "proyecto-integrador/02_tratamiento/andinalog_flota_eventos/salidas/andinalog_flota_eventos_silver.csv"
SALIDAS = RAIZ / "proyecto-integrador/03_EDA/salidas_v4_eventos"

iot = pd.read_csv(RUTA_IOT, encoding="utf-8-sig")
eventos = pd.read_csv(RUTA_EVENTOS, encoding="utf-8-sig")
iot["timestamp_bolivia"] = pd.to_datetime(iot["timestamp_bolivia"], errors="raise")
eventos["timestamp_bolivia"] = pd.to_datetime(eventos["timestamp_bolivia"], errors="raise")
eventos["ultimo_mantenimiento"] = pd.to_datetime(eventos["ultimo_mantenimiento"], errors="raise")
assert not eventos["evento_id"].duplicated().any()
assert eventos["camion_id"].isin(iot["camion_id_tratado"]).all()
print("IoT:", iot.shape, "Eventos:", eventos.shape)


## Construcción de variables

Las variables resumen frecuencia, tipo, severidad, reconocimiento, recencia y configuración. `valor_lectura_numerico` queda fuera porque su unidad depende del tipo y no está declarada.

In [ ]:
# Una configuración por camión; se valida antes de difundirla a las lecturas.
config_cols = ["umbral_temp_cabina_c", "geocerca_radio_km", "ultimo_mantenimiento"]
variaciones = eventos.groupby("camion_id")[config_cols].nunique(dropna=False)
assert variaciones.le(1).all().all(), "La configuración cambia dentro de un camión"
config = eventos.groupby("camion_id", as_index=True)[config_cols].first()

resultado = iot.copy()
features_conteo = [
    "eventos_previos_60m", "eventos_previos_180m", "eventos_previos_24h",
    "alertas_temp_previas_60m", "alertas_temp_previas_180m", "alertas_temp_previas_24h",
    "fallas_motor_previas_24h", "eventos_alta_previos_24h",
    "eventos_no_reconocidos_previos_24h", "reconocimiento_faltante_previos_24h"]
for c in features_conteo:
    resultado[c] = 0
resultado["minutos_desde_ultimo_evento"] = np.nan
resultado["minutos_desde_ultima_alerta_temp"] = np.nan

def conteo_ventana(t_eventos, t_lecturas, ventana_min, mascara=None):
    izq = np.searchsorted(t_eventos, t_lecturas - np.timedelta64(ventana_min, "m"), side="left")
    der = np.searchsorted(t_eventos, t_lecturas, side="left")  # estrictamente anteriores
    if mascara is None:
        return der - izq
    acumulada = np.r_[0, np.cumsum(mascara.astype(int))]
    return acumulada[der] - acumulada[izq]

for camion, idx_lecturas in resultado.groupby("camion_id_tratado").groups.items():
    ev = eventos.loc[eventos["camion_id"].eq(camion)].sort_values("timestamp_bolivia").copy()
    idx = np.array(list(idx_lecturas))
    orden = np.argsort(resultado.loc[idx, "timestamp_bolivia"].to_numpy())
    idx_ord = idx[orden]
    tr = resultado.loc[idx_ord, "timestamp_bolivia"].to_numpy(dtype="datetime64[ns]")
    te = ev["timestamp_bolivia"].to_numpy(dtype="datetime64[ns]")
    tipo = ev["tipo"].to_numpy()
    severidad = ev["severidad"].to_numpy()
    reconocido = ev["reconocido"].astype("string").str.lower().fillna("faltante").to_numpy(dtype=str)
    rec_faltante = ev["reconocido_faltante"].astype(bool).to_numpy()

    for minutos, sufijo in [(60, "60m"), (180, "180m"), (1440, "24h")]:
        resultado.loc[idx_ord, f"eventos_previos_{sufijo}"] = conteo_ventana(te, tr, minutos)
        resultado.loc[idx_ord, f"alertas_temp_previas_{sufijo}"] = conteo_ventana(te, tr, minutos, tipo == "ALERTA_TEMP_CADENA_FRIO")
    resultado.loc[idx_ord, "fallas_motor_previas_24h"] = conteo_ventana(te, tr, 1440, tipo == "FALLA_MOTOR")
    resultado.loc[idx_ord, "eventos_alta_previos_24h"] = conteo_ventana(te, tr, 1440, severidad == "Alta")
    resultado.loc[idx_ord, "eventos_no_reconocidos_previos_24h"] = conteo_ventana(te, tr, 1440, reconocido == "false")
    resultado.loc[idx_ord, "reconocimiento_faltante_previos_24h"] = conteo_ventana(te, tr, 1440, rec_faltante)

    pos = np.searchsorted(te, tr, side="left") - 1
    valido = pos >= 0
    minutos = np.full(len(tr), np.nan)
    minutos[valido] = (tr[valido] - te[pos[valido]]) / np.timedelta64(1, "m")
    resultado.loc[idx_ord, "minutos_desde_ultimo_evento"] = minutos
    te_alerta = te[tipo == "ALERTA_TEMP_CADENA_FRIO"]
    pos_a = np.searchsorted(te_alerta, tr, side="left") - 1
    valido_a = pos_a >= 0
    minutos_a = np.full(len(tr), np.nan)
    minutos_a[valido_a] = (tr[valido_a] - te_alerta[pos_a[valido_a]]) / np.timedelta64(1, "m")
    resultado.loc[idx_ord, "minutos_desde_ultima_alerta_temp"] = minutos_a

resultado["umbral_eventos_temp_cabina_c"] = resultado["camion_id_tratado"].map(config["umbral_temp_cabina_c"])
resultado["geocerca_eventos_radio_km"] = resultado["camion_id_tratado"].map(config["geocerca_radio_km"])
resultado["ultimo_mantenimiento_eventos"] = resultado["camion_id_tratado"].map(config["ultimo_mantenimiento"])
resultado["dias_desde_ultimo_mantenimiento_eventos"] = (
    resultado["timestamp_bolivia"] - resultado["ultimo_mantenimiento_eventos"]
).dt.total_seconds().div(86400)
resultado["tiene_evento_previo_24h"] = resultado["eventos_previos_24h"].gt(0)
resultado["tiene_alerta_temp_previa_24h"] = resultado["alertas_temp_previas_24h"].gt(0)

# Pruebas de integridad y ausencia de fuga temporal.
assert len(resultado) == len(iot)
assert resultado["fila_bronze"].tolist() == iot["fila_bronze"].tolist()
assert resultado[features_conteo].ge(0).all().all()
assert resultado["eventos_previos_60m"].le(resultado["eventos_previos_180m"]).all()
assert resultado["eventos_previos_180m"].le(resultado["eventos_previos_24h"]).all()
assert resultado["minutos_desde_ultimo_evento"].dropna().gt(0).all()
assert resultado["umbral_eventos_temp_cabina_c"].notna().all()
print("Integración temporal terminada sin multiplicación de filas")


## Auditoría y exportación

Se comprueba conservación exacta de las 28.677 lecturas, monotonía entre ventanas y temporalidad estrictamente anterior.

In [ ]:
# Resumen del aporte de las nuevas variables.
resumen = {
    "lecturas_iot_entrada": len(iot),
    "lecturas_salida": len(resultado),
    "eventos_silver": len(eventos),
    "camiones_iot": iot["camion_id_tratado"].nunique(),
    "camiones_eventos": eventos["camion_id"].nunique(),
    "cobertura_configuracion_pct": round(resultado["umbral_eventos_temp_cabina_c"].notna().mean() * 100, 2),
    "lecturas_con_evento_previo_60m": int(resultado["eventos_previos_60m"].gt(0).sum()),
    "lecturas_con_evento_previo_180m": int(resultado["eventos_previos_180m"].gt(0).sum()),
    "lecturas_con_evento_previo_24h": int(resultado["eventos_previos_24h"].gt(0).sum()),
    "lecturas_con_alerta_temp_previa_24h": int(resultado["alertas_temp_previas_24h"].gt(0).sum()),
}

comparacion = resultado.groupby("tiene_evento_previo_24h", dropna=False).agg(
    lecturas=("fila_bronze", "size"),
    tasa_desviacion_futura=("clasificacion_objetivo_60min", "mean"),
    desvio_futuro_promedio_c=("max_desvio_termico_proximos_60min_c", "mean")
).reset_index()
print(resumen)
print(comparacion.to_string(index=False))

SALIDAS.mkdir(parents=True, exist_ok=True)
resultado_exportar = resultado.copy()
resultado_exportar["timestamp_bolivia"] = resultado_exportar["timestamp_bolivia"].dt.strftime("%Y-%m-%d %H:%M:%S")
resultado_exportar["ultimo_mantenimiento_eventos"] = resultado_exportar["ultimo_mantenimiento_eventos"].dt.strftime("%Y-%m-%d")
resultado_exportar.to_csv(SALIDAS / "andinalog_03b_evidencia_3_lecturas_con_eventos.csv", index=False, encoding="utf-8-sig")
pd.DataFrame([{"metrica": k, "valor": v} for k, v in resumen.items()]).to_csv(
    SALIDAS / "andinalog_03b_integracion_eventos_resumen.csv", index=False, encoding="utf-8-sig")


## Interpretación

Las diferencias observadas entre grupos son asociaciones exploratorias. La utilidad predictiva de los eventos se comprobará contra un baseline durante S6 y S7.